# 03 — Goal 2: participant persistence

Quantifies repeated-recording variance structure while explicitly avoiding a test–retest reliability claim.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


In [ ]:
RUN_GOAL_2 = False
if RUN_GOAL_2:
    run_cli("describe")

from paper1_qc.registry import metric_registry_frame
data = read_stage("03_dataset_assembly/paper1_analysis_dataset")
data = data.loc[data["primary_measurement_eligible"].fillna(False)].copy()
persistence = read_stage("04_analysis/descriptive/participant_persistence_not_reliability")
registry = metric_registry_frame()
persistence = persistence.merge(
    registry[["feature", "family", "unit", "role"]],
    on="feature", how="left", validate="one_to_one"
)


In [ ]:
repeat_counts = (
    data.groupby("SubjectID").size().rename("recordings").reset_index()
)
repeat_summary = repeat_counts["recordings"].value_counts().sort_index().rename_axis(
    "recordings_per_participant"
).rename("participants").reset_index()
repeat_summary["participant_fraction"] = repeat_summary["participants"] / len(repeat_counts)
save_table(repeat_summary, "03_goal2", "repeated_recording_design")
display(repeat_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=repeat_summary, x="recordings_per_participant", y="participants", ax=ax)
ax.set(title="Repeated-recording support", xlabel="Eligible recordings per participant", ylabel="Participants")
save_figure(fig, "03_goal2", "recordings_per_participant")
plt.show()


In [ ]:
status_summary = (
    persistence.groupby(["family", "status"], dropna=False)
    .size().rename("metrics").reset_index()
)
save_table(status_summary, "03_goal2", "persistence_model_status")
display(status_summary)

estimable = persistence.loc[persistence["status"].isin(["ok", "not_converged"])].copy()
estimable = estimable.sort_values(["family", "persistence_icc"])
fig, ax = plt.subplots(figsize=(11, max(5, .35 * len(estimable))))
sns.scatterplot(
    data=estimable,
    x="persistence_icc",
    y="feature",
    hue="family",
    s=70,
    ax=ax,
)
ax.axvline(.5, color="0.4", linestyle="--", linewidth=1)
ax.set(
    title="Participant rank persistence (not test–retest reliability)",
    xlabel="Between-participant variance / total variance",
    ylabel="",
    xlim=(-.05, 1.05),
)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
save_figure(fig, "03_goal2", "participant_rank_persistence")
plt.show()


In [ ]:
# Auditable spaghetti plots for selected high-, middle-, and low-persistence metrics.
if len(estimable):
    ordered = estimable.sort_values("persistence_icc")
    selected_features = list(dict.fromkeys([
        ordered.iloc[0]["feature"],
        ordered.iloc[len(ordered)//2]["feature"],
        ordered.iloc[-1]["feature"],
    ]))
else:
    selected_features = []

date_column = "Recording date"
for feature in selected_features:
    work = data[["SubjectID", date_column, feature]].copy()
    work[date_column] = pd.to_datetime(work[date_column], errors="coerce")
    work[feature] = pd.to_numeric(work[feature], errors="coerce")
    work = work.dropna().sort_values(["SubjectID", date_column])
    work["occasion"] = work.groupby("SubjectID").cumcount() + 1
    repeated_ids = work.groupby("SubjectID").size().loc[lambda x: x >= 2].index
    work = work.loc[work["SubjectID"].isin(repeated_ids)]
    # Deterministic display subset only; the model used all eligible repeated participants.
    display_ids = sorted(repeated_ids.astype(str))[:40]
    plot_data = work.loc[work["SubjectID"].astype(str).isin(display_ids)]
    fig, ax = plt.subplots(figsize=(10, 5))
    for _, subject in plot_data.groupby("SubjectID"):
        ax.plot(subject["occasion"], subject[feature], color="0.35", alpha=.35, linewidth=.8)
    ax.set(title=f"Repeated-recording trajectories: {feature}", xlabel="Observed occasion", ylabel=feature)
    save_figure(fig, "03_goal2", f"spaghetti__{feature}")
    plt.show()


In [ ]:
# Between- vs within-participant variance table preserves the actual estimand.
variance_table = persistence[[
    "feature", "family", "n_recordings", "n_participants", "n_repeated_participants",
    "between_participant_variance", "within_participant_variance", "persistence_icc",
    "zero_fraction", "status",
]].copy()
save_table(variance_table, "03_goal2", "participant_persistence_full")
display(variance_table)
